In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import flwr as fl

from torch.utils.data import DataLoader, TensorDataset, random_split
# Use a held-out validation split for federated evaluation

ARTIFACT_DIR = "./artifacts"

NUM_SCHOOLS = 3
SAMPLES_PER_SCHOOL = 300

LOCAL_EPOCHS = 3
BATCH_SIZE = 32
LR = 0.01
VAL_RATIO = 0.2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

Device: cuda


In [2]:
def generate_learning_dataset(school_id, n_samples=300):

    rng = np.random.default_rng(100 + school_id)

    reading = rng.normal(150,40,n_samples)
    grammar = rng.normal(40,15,n_samples)
    vocab = rng.normal(60,20,n_samples)
    quiz = rng.normal(65,12,n_samples)
    completion = rng.normal(0.7,0.15,n_samples)

    reading = np.clip(reading,0,300)
    grammar = np.clip(grammar,0,80)
    vocab = np.clip(vocab,0,150)
    quiz = np.clip(quiz,0,100)
    completion = np.clip(completion,0,1)

    features = np.stack([
        reading,
        grammar,
        vocab,
        quiz,
        completion
    ],axis=1)

    labels = []

    for r,g,v,q,c in features:

        if q < 60:
            labels.append(3)  # quiz practice
        elif g < 30:
            labels.append(1)  # grammar
        elif v < 40:
            labels.append(2)  # vocab
        else:
            labels.append(0)  # reading

    df = pd.DataFrame(features,columns=[
        "reading_minutes",
        "grammar_attempts",
        "vocab_attempts",
        "quiz_score",
        "completion_rate"
    ])

    df["recommended_activity"] = labels

    return df

In [3]:
for i in range(NUM_SCHOOLS):

    df = generate_learning_dataset(i,SAMPLES_PER_SCHOOL)

    path = f"{ARTIFACT_DIR}/flrec_school_{i+1}_dataset.csv"

    df.to_csv(path,index=False)

    print("Saved:",path)

Saved: ./artifacts/flrec_school_1_dataset.csv
Saved: ./artifacts/flrec_school_2_dataset.csv
Saved: ./artifacts/flrec_school_3_dataset.csv


In [6]:
def load_dataset(path):

    df = pd.read_csv(path)

    X = df[[
        "reading_minutes",
        "grammar_attempts",
        "vocab_attempts",
        "quiz_score",
        "completion_rate"
    ]].values.astype(np.float32)

    y = df["recommended_activity"].values.astype(np.int64)

    X = torch.tensor(X)
    y = torch.tensor(y)

    dataset = TensorDataset(X,y)

    val_size = max(1, int(len(dataset) * VAL_RATIO))
    train_size = len(dataset) - val_size

    split_generator = torch.Generator().manual_seed(42)
    train_dataset, val_dataset = random_split(
        dataset,
        [train_size, val_size],
        generator=split_generator
    )

    trainloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    valloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

    return trainloader, valloader

In [7]:
class LearningRecommender(nn.Module):

    def __init__(self):

        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(5,32),
            nn.ReLU(),
            nn.Linear(32,16),
            nn.ReLU(),
            nn.Linear(16,4)
        )

    def forward(self,x):
        return self.net(x)

In [8]:
def get_parameters(model):
    return [val.cpu().numpy() for _,val in model.state_dict().items()]

def set_parameters(model,parameters):

    params_dict = zip(model.state_dict().keys(),parameters)

    state_dict = {k:torch.tensor(v) for k,v in params_dict}

    model.load_state_dict(state_dict)

In [9]:
class SchoolClient(fl.client.NumPyClient):

    def __init__(self,model,trainloader,valloader):

        self.model = model
        self.trainloader = trainloader
        self.valloader = valloader

    def get_parameters(self,config):
        return get_parameters(self.model)

    def fit(self,parameters,config):

        set_parameters(self.model,parameters)

        optimizer = optim.Adam(self.model.parameters(),lr=LR)
        loss_fn = nn.CrossEntropyLoss()

        self.model.train()

        for _ in range(LOCAL_EPOCHS):

            for X,y in self.trainloader:

                X,y = X.to(device),y.to(device)

                optimizer.zero_grad()

                preds = self.model(X)

                loss = loss_fn(preds,y)

                loss.backward()

                optimizer.step()

        return get_parameters(self.model),len(self.trainloader.dataset),{}

    def evaluate(self,parameters,config):

        set_parameters(self.model,parameters)

        self.model.eval()
        # Report validation loss instead of a hardcoded 0.0
        loss_fn = nn.CrossEntropyLoss(reduction="sum")

        total_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():

            for X,y in self.valloader:

                X,y = X.to(device),y.to(device)

                preds = self.model(X)

                total_loss += loss_fn(preds,y).item()

                predicted = torch.argmax(preds,dim=1)

                correct += (predicted==y).sum().item()

                total += y.size(0)

        average_loss = total_loss/total
        accuracy = correct/total

        return average_loss,total,{"accuracy":accuracy}

In [10]:
def client_fn(cid):

    model = LearningRecommender().to(device)

    path = f"{ARTIFACT_DIR}/flrec_school_{int(cid)+1}_dataset.csv"

    trainloader, valloader = load_dataset(path)

    return SchoolClient(model,trainloader,valloader)

In [9]:
strategy = fl.server.strategy.FedAvg(
    fraction_fit=1.0,
    min_fit_clients=NUM_SCHOOLS,
    min_available_clients=NUM_SCHOOLS
)

fl.simulation.start_simulation(
    client_fn=client_fn,
    num_clients=NUM_SCHOOLS,
    config=fl.server.ServerConfig(num_rounds=10),
    strategy=strategy
)

	Instead, use the `flwr run` CLI command to start a local simulation in your Flower app, as shown for example below:

		$ flwr new  # Create a new Flower app from a template

		$ flwr run  # Run the Flower app in Simulation Mode

	Using `start_simulation()` is deprecated.

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
INFO :      Starting Flower simulation, config: num_rounds=10, no round_timeout
2026-03-14 14:08:06,154	INFO worker.py:2012 -- Started a local Ray instance.
c:\Users\wassi\.conda\envs\pytorch-benchmark-winmac\lib\site-packages\ray\_private\worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(
INFO :      Flower VCE: Ray initialized with resources: {'object_store_memory': 2

History (loss, distributed):
	round 1: 0.0
	round 2: 0.0
	round 3: 0.0
	round 4: 0.0
	round 5: 0.0
	round 6: 0.0
	round 7: 0.0
	round 8: 0.0
	round 9: 0.0
	round 10: 0.0

(pid=gcs_server) [2026-03-14 14:08:35,988 E 30888 28392] (gcs_server.exe) gcs_server.cc:302: Failed to establish connection to the event+metrics exporter agent. Events and metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14
(raylet) [2026-03-14 14:08:37,903 E 29928 25616] (raylet.exe) main.cc:975: Failed to establish connection to the metrics exporter agent. Metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14
